In [2]:
!pip install torch torchvision

  Using cached pillow-12.1.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 35.4 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 36.0 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 36.5 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 36.5 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 37.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 35.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 34.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 36.2 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 25.6 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 37.3 MB/s  0:00:02m0:00:0100:01
   ━━━━

In [ ]:
"""
emotion_efficientnet.py  –  FER2013 emotion recogniser
Uses a pretrained EfficientNet-B0 fine-tuned on FER2013.
Expected accuracy: 68–72% on FER2013 test set.
Requirements: torch torchvision opencv-python scikit-learn
"""

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib

# ── Config ────────────────────────────────────────────────────
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 224      # EfficientNet expects 224×224
BATCH_SIZE = 32       # smaller than before — 224px images are heavier
EPOCHS     = 10
LR_HEAD    = 1e-3     # classifier head (randomly initialised)
LR_BODY    = 1e-4     # pretrained backbone (fine-tuning, slower)
MODEL_NAME = "meow"
os.makedirs("models", exist_ok=True)

# ── 1. Dataset ────────────────────────────────────────────────
class FERDataset(Dataset):
    def __init__(self, root: str, label_encoder: LabelEncoder,
                 transform=None, fit_encoder: bool = False):
        self.transform = transform
        self.samples: list[tuple[str, str]] = []

        for emotion in sorted(os.listdir(root)):
            folder = os.path.join(root, emotion)
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                path = os.path.join(folder, fname)
                if cv2.imread(path, cv2.IMREAD_GRAYSCALE) is not None:
                    self.samples.append((path, emotion))

        paths, labels = zip(*self.samples)
        if fit_encoder:
            label_encoder.fit(labels)
        self.labels = label_encoder.transform(labels)
        self.paths  = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx], cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        # EfficientNet was pretrained on RGB — replicate the single channel
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ImageNet normalisation — required because the backbone weights expect it
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

le = LabelEncoder()
train_ds = FERDataset("../datasets/fer2013/train", le, train_tf, fit_encoder=True)
test_ds  = FERDataset("../datasets/fer2013/test",  le, test_tf,  fit_encoder=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

NUM_CLASSES = len(le.classes_)
print(f"Classes : {le.classes_}")
print(f"Train   : {len(train_ds)}   Test: {len(test_ds)}   Device: {DEVICE}")

# ── 2. Model ──────────────────────────────────────────────────
def build_model(num_classes: int) -> nn.Module:
    """
    EfficientNet-B0 pretrained on ImageNet.
    We replace the final classifier layer with one sized for our emotions.
    The backbone is kept frozen for the first few epochs, then unfrozen.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Freeze all backbone parameters initially
    for param in model.parameters():
        param.requires_grad = False

    # Replace the classifier head (1280 → num_classes)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )
    # Head params are trainable from the start
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


model = build_model(NUM_CLASSES).to(DEVICE)

head_params = list(model.classifier.parameters())
body_params = [p for p in model.features.parameters()]  # frozen initially

# ── 3. Training — two-phase ───────────────────────────────────
#
# Phase 1 (epochs 1–5):  only the new head is trained.
#   The pretrained backbone is frozen so its ImageNet features are preserved
#   while the head converges to sensible emotion predictions.
#
# Phase 2 (epochs 6–20): unfreeze the whole backbone with a lower LR.
#   Fine-tuning adjusts the features to facial expression structure.

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
best_acc   = 0.0
best_path  = f"models/emotion_efficientnet_{MODEL_NAME}_best.pt"

UNFREEZE_EPOCH = 3   # switch from phase 1 → phase 2 here

# Start with only head parameters in the optimiser
optimizer = optim.AdamW(head_params, lr=LR_HEAD, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):

    # ── Phase transition: unfreeze backbone ──
    if epoch == UNFREEZE_EPOCH + 1:
        print("\n── Unfreezing backbone for fine-tuning ──")
        for param in model.features.parameters():
            param.requires_grad = True
        # Re-create optimiser with two param groups at different LRs
        optimizer = optim.AdamW([
            {"params": body_params, "lr": LR_BODY},
            {"params": head_params, "lr": LR_HEAD * 0.1},
        ], weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - UNFREEZE_EPOCH
        )

    # ── Train ──
    model.train()
    running_loss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    scheduler.step()
    avg_loss = running_loss / len(train_ds)

    # ── Validate ──
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    acc = correct / total

    phase = "head-only" if epoch <= UNFREEZE_EPOCH else "fine-tune"
    print(f"Epoch {epoch:02d}/{EPOCHS} [{phase}]  "
          f"loss={avg_loss:.4f}  test_acc={acc:.1%}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_path)
        print(f"  ↳ saved best model ({best_acc:.1%})")

# ── 4. Final evaluation ───────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_dl:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print("\n" + classification_report(all_labels, all_preds, target_names=le.classes_))

joblib.dump(le, f"models/emotion_efficientnet_labels_{MODEL_NAME}.pkl")
print("Label encoder saved.")

# ── 5. Single-image inference ─────────────────────────────────
def predict_emotion(img_path: str, model=model, le=le) -> str:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    tensor = test_tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        idx = model(tensor).argmax(1).item()
    return le.inverse_transform([idx])[0]

# ── 6. Load saved model (in a separate script) ────────────────
def load_model(model_name: str = MODEL_NAME):
    _le    = joblib.load(f"models/emotion_efficientnet_labels_{model_name}.pkl")
    _model = build_model(len(_le.classes_)).to(DEVICE)
    _model.load_state_dict(
        torch.load(f"models/emotion_efficientnet_{model_name}_best.pt",
                   map_location=DEVICE)
    )
    _model.eval()
    return _model, _le


# Quick smoke-test (uncomment to verify after training)
# m, enc = load_model()
# print(predict_emotion("some_face.jpg", model=m, le=enc))

Classes : ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']
Train   : 28709   Test: 7178   Device: cuda
Epoch 01/10 [head-only]  loss=1.7109  test_acc=36.9%
  ↳ saved best model (36.9%)
Epoch 02/10 [head-only]  loss=1.6718  test_acc=39.0%
  ↳ saved best model (39.0%)
Epoch 03/10 [head-only]  loss=1.6577  test_acc=39.4%
  ↳ saved best model (39.4%)

── Unfreezing backbone for fine-tuning ──
Epoch 04/10 [fine-tune]  loss=1.4351  test_acc=58.8%
  ↳ saved best model (58.8%)
Epoch 05/10 [fine-tune]  loss=1.2813  test_acc=63.1%
  ↳ saved best model (63.1%)
Epoch 06/10 [fine-tune]  loss=1.2060  test_acc=65.0%
  ↳ saved best model (65.0%)
Epoch 07/10 [fine-tune]  loss=1.1504  test_acc=66.5%
  ↳ saved best model (66.5%)
